# 🚀 Muse Spark Open Kernel Challenge ($100k)
### Official Submission Runner for NVIDIA H200 (Hopper SM90a)
**Participant:** Arunjyoti Sarma (IIT BHU Research Lab)  
**Target Problem:** `muse_bf16_proj` (Dense Projections)

This notebook completely automates the entire Open Kernel Challenge workflow on Google Colab.

## Step 1: Verify Cloud GPU
Click the Play button below to verify your connected remote GPU.

In [ ]:
!nvidia-smi

## Step 2: 1-Click Complete Pipeline (Install, Login, Deploy & Submit!)

Click the Play button on the cell below:
1. It will install `okc` CLI.
2. It will display your 8-character device code in the output.
3. Switch to your **[github.com/login/device](https://github.com/login/device)** tab, enter the code, and click **Authorize**.
4. The cell will automatically detect your login, deploy our optimized zero-allocation Hopper H200 kernel into `muse_bf16_proj/submission.py`, and submit directly to Meta's 8× H200 SXM cluster!

In [12]:
!/root/.okc/bin/okc leaderboard muse_bf16_proj

◆ muse_bf16_proj  time per decode step, lower is better

  #   NAME                                   SPEEDUP
────────────────────────────────────────────────────
  1   gogowhy                            S    1.694×
  2   dolanzhao                               1.687×
  3   reyanshsolis                            1.684×
  4   olety                              F    1.680×
  5   gauravjain14                       S    1.671×
  6   jt914                                   1.661×
  7   tjiang-fair                        F    1.660×
  8   sudodaksh                          S    1.653×
  9   Fangzhou66                              1.647×
 10   miketung                           S    1.646×
 11   mgpai22                            S    1.646×
 12   YounesElhjouji                     S    1.642×
 13   pjmullin3                          S    1.639×
 14   arashattar                              1.637×
 15   colinlikescode                     S    1.637×
 16   DTennant                            

In [13]:
!/root/.okc/bin/okc status a03d889ec6734035


✓ muse_bf16_proj  passed  job a03d889ec6734035

  1.492×  ━━━━━━━━━╋━━━━━─────────────  faster than SGLang  95%: 1.470–1.515×

  time        3,237.4 µs
  tests       9/9 passed
  profile     profiles/a03d889ec6734035.txt  and .json
  output      profiles/a03d889ec6734035.log

  CASE                    SGLANG     YOURS   SPEEDUP   
  ─────────────────────────────────────────────────────────────────
  decode 1 · qkv_gate    14.0 µs    8.9 µs    1.567×   ━━━━━━╋━━━──
  decode 1 · o            6.6 µs    5.0 µs    1.320×   ━━━━━━╋━────
  decode 1 · pooler      21.0 µs   13.2 µs    1.588×   ━━━━━━╋━━━──
  decode 1 · unpooler     7.2 µs    3.7 µs    1.943×   ━━━━━━╋━━━━━
  decode 1 · router      10.4 µs    3.7 µs    2.834×   ━━━━━━╋━━━━━
  decode 16 · qkv_gate   14.2 µs    9.9 µs    1.438×   ━━━━━━╋━━───
  decode 16 · o           6.6 µs    5.4 µs    1.236×   ━━━━━━╋━────
  decode 16 · pooler     21.5 µs   14.2 µs    1.521×   ━━━━━━╋━━━──
  decode 16 · unpooler   18.5 µs   13.6 µs    1.357×   

In [5]:

    import os, urllib.request

    os.makedirs("muse_bf16_proj", exist_ok=True)

    url = (
        "https://raw.githubusercontent.com/"
        "arunjyotisarmarsece24-jpg/"
        "neuro-spark-accel/main/colab_submission.py"
    )
    dst = "muse_bf16_proj/submission.py"

    urllib.request.urlretrieve(url, dst)
    print("✓ Kernel downloaded. Size:", os.path.getsize(dst), "bytes")

    os.system("/root/.okc/bin/okc submit muse_bf16_proj/submission.py")


✓ Kernel downloaded. Size: 195623 bytes


32512

In [6]:
 !/root/.okc/bin/okc submit muse_bf16_proj/submission.py

/bin/bash: line 1: /root/.okc/bin/okc: No such file or directory


In [7]:
 !curl -fsSL https://openkernelchallenge.dev/install.sh | sh

In [8]:
!/root/.okc/bin/okc submit muse_bf16_proj/submission.py

✗ not signed in

next  okc login


In [9]:

    import os, pty, select, subprocess, time, urllib.request

    os.environ["PATH"] = "/root/.okc/bin:" + os.environ.get("PATH", "")

    # 1. GitHub Device Authentication
    config_path = "/root/.config/okc/config.json"
    if not os.path.exists(config_path):
      print("[*] Starting GitHub Device Flow for authentication...\n")
      master, slave = pty.openpty()
      proc = subprocess.Popen(
          ["/root/.okc/bin/okc", "login"],
          stdin=slave,
          stdout=slave,
          stderr=slave,
          close_fds=True,
      )
      os.close(slave)
      time.sleep(1.0)
      os.write(master, b"y\n")

      while True:
        r, _, _ = select.select([master], [], [], 2.0)
        if r:
          try:
            data = os.read(master, 1024)
            if not data:
              break
            text = data.decode("utf-8", errors="ignore")
            print(text, end="", flush=True)
            if "token saved" in text:
              break
          except OSError:
            break
        elif proc.poll() is not None:
          break
      print(
          "\n👉 Open https://github.com/login/device, enter the code above, and"
          " click Authorize!"
      )
      proc.wait()
      print("\n✓ Authentication completed!")
    else:
      print("✓ Already authenticated!")

    # 2. Download our new 195 KB kernel (9-stage TMA + zero-latency dispatch)
    os.makedirs("muse_bf16_proj", exist_ok=True)
    url = "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/colab_submission.py"
    dst = "muse_bf16_proj/submission.py"
    urllib.request.urlretrieve(url, dst)
    print("✓ Kernel downloaded:", os.path.getsize(dst), "bytes")

    # 3. Submit directly to Meta's H200 cluster and live-stream output
    print(
        "\n[🚀] Submitting to Meta H200 cluster & leaderboard...\n" + "=" * 60 + "\n"
    )
    sub = subprocess.Popen(
        ["/root/.okc/bin/okc", "submit", dst],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in sub.stdout:
      print(line, end="", flush=True)
    sub.wait()

[*] Starting GitHub Device Flow for authentication...

◆ okc · open kernel challenge · muse spark open
  meta × paradigm × andromeda

y
Participation is governed by the Paradigm Muse Spark Open Kernel Challenge Official Rules and
Paradigm Terms of Use:
  Official Rules   https://openkernelchallenge.dev/terms.pdf
  Terms of Use     https://www.paradigm.xyz/terms
The Paradigm Privacy Policy applies to your information:
  Privacy Policy   https://www.paradigm.xyz/privacy

Do you agree to the Challenge Rules, Paradigm Terms, and Paradigm Privacy Policy? [y/N] 
╭───────────────╮
│   your code   │
│   96AE-B0D4   │
╰───────────────╯

› enter it at https://github.com/login/device
› opened in your browser
✓ signed in as arunjyotisarmarsece24-jpg
  token saved to /root/.config/okc/config.json

Do you agree that Paradigm may disclose your email address to Meta for its recruiting and business purposes? [y/N] 
👉 Open https://github.com/login/device, enter the code above, and click Authorize!


KeyboardInterrupt: 

In [11]:
   import urllib.request

    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/colab_submission.py",
        "muse_bf16_proj/submission.py",
    )
    print("✓ Downloaded!")

IndentationError: unexpected indent (619286882.py, line 3)

In [12]:
 /root/.okc/bin/okc submit muse_bf16_proj/submission.py


SyntaxError: invalid syntax (1981461203.py, line 1)

In [13]:
 !/root/.okc/bin/okc submit muse_bf16_proj/submission.py

Streaming output truncated to the last 5000 lines.
✓ queued for a GPU  7:00
⠇ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠏ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠋ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠙ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠹ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠸ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠼ running the tests, then timing against SGLang  3:47


✓ received  1:58
✓ queued for a GPU  7:00
⠴ running the tests, then timing against SGLang  3:48


✓ received  1:58
✓ queued for a GPU  7:00
⠦ running the tests, then timing against SGLang  3:48


✓ received  1:58
✓ queued for a GPU  7:00
⠧ running the tests, then timing against S

In [3]:
    import os

    os.system(
        "curl -fsSL"
        " https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/colab_submission.py"
        " -o muse_bf16_proj/submission.py"
    )
    os.system("/root/.okc/bin/okc submit muse_bf16_proj/submission.py")

32512

In [7]:
 !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash

=== [1/3] Preparing workspace ===
=== [2/3] Downloading latest 199 KB Beast Kernel (commit c6faf91) ===
✓ Kernel ready! Size: 195620 bytes
=== [3/3] Submitting to Meta H200 Cluster ===
bash: line 14: /root/.okc/bin/okc: No such file or directory


In [8]:
 !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash

=== [1/4] Ensuring OKC CLI is installed ===
Installing okc CLI...


In [9]:
 !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash

=== [1/4] Ensuring OKC CLI is installed ===
✓ OKC CLI found: /root/.okc/bin/okc
=== [2/4] Preparing workspace ===
=== [3/4] Downloading latest 199 KB Beast Kernel (commit c6faf91) ===
✓ Kernel ready! Size: 195620 bytes
=== [4/4] Submitting to Meta H200 Cluster ===
✗ not signed in

next  okc login


In [10]:
 !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash

🚀 OKC LEADERBOARD RUNNER - NVIDIA H200 SM90a

[2/4] Authenticating with GitHub Device Flow...
◆ okc · open kernel challenge · muse spark open
  meta × paradigm × andromeda

Participation is governed by the Paradigm Muse Spark Open Kernel Challenge Official Rules and
Paradigm Terms of Use:
  Official Rules   https://openkernelchallenge.dev/terms.pdf
  Terms of Use     https://www.paradigm.xyz/terms
The Paradigm Privacy Policy applies to your information:
  Privacy Policy   https://www.paradigm.xyz/privacy

Do you agree to the Challenge Rules, Paradigm Terms, and Paradigm Privacy Policy? [y/N] y

╭───────────────╮
│   your code   │
│   5438-FCA0   │
╰───────────────╯

› enter it at https://github.com/login/device
› opened in your browser
✓ signed in as arunjyotisarmarsece24-jpg
  token saved to /root/.config/okc/config.json

Do you agree that Paradigm may disclose your email address to Meta for its recruiting and business purposes? [y/N] 
✓ Auth token backed up successfully!

[3/4] Downl

In [11]:
!/root/.okc/bin/okc leaderboard muse_bf16_proj

◆ muse_bf16_proj  time per decode step, lower is better

  #   NAME                                   SPEEDUP
────────────────────────────────────────────────────
  1   gogowhy                            S    1.694×
  2   olety                                   1.688×
  3   dolanzhao                               1.687×
  4   reyanshsolis                            1.684×
  5   gauravjain14                       S    1.671×
  6   jt914                                   1.662×
  7   tjiang-fair                        F    1.660×
  8   sudodaksh                          S    1.653×
  9   arashattar                              1.650×
 10   Fangzhou66                              1.647×
 11   miketung                           S    1.646×
 12   mgpai22                            S    1.646×
 13   YounesElhjouji                     S    1.642×
 14   pjmullin3                          S    1.639×
 15   colinlikescode                     S    1.637×
 16   ultramathi                          

In [12]:
 !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash

🚀 OKC LEADERBOARD RUNNER - NVIDIA H200 SM90a

[2/4] ✓ Already signed into OKC!

[3/4] Downloading latest 199 KB Beast Kernel (commit c6faf91)...
✓ Kernel ready: 197,111 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job a5e09aea439d4403
› received
› received  0:30
› received  1:00
› received  1:30
› queued for a GPU  1:57
› queued for a GPU  0:03
› queued for a GPU  0:33
› queued for a GPU  1:03
› queued for a GPU  1:33
› queued for a GPU  2:03
› queued for a GPU  2:33
› queued for a GPU  3:03
› queued for a GPU  3:33
› queued for a GPU  4:03
› queued for a GPU  4:33
› queued for a GPU  5:03
› queued for a GPU  5:33
› queued for a GPU  6:03
› queued for a GPU  6:33
› queued for a GPU  7:03
› running the tests, then timing against SGLang  9:10
› running the tests, then timing against SGLang  0:20
› running the tests, then timing against SGLang  0:50
› running the tests, then timing against SGLang  1:20
› running

In [13]:
 !python3 -c "import json, glob, os; p=sorted(glob.glob('muse_bf16_proj/profiles/*.json'), key=os.path.getmtime, reverse=True); print(json.dumps(json.load(open(p[0])), indent=2) if p else 'no profile')"

{
  "benchmarks": [
    {
      "baseline_ns": 14240.000862628222,
      "baseline_parent_ns": 21311.987657099962,
      "baseline_raw_ns": 18208.00080895424,
      "baseline_runs_ns": [
        18880,
        18208,
        18112,
        18048,
        17760,
        17312,
        18048,
        17920,
        18240,
        17920,
        18304,
        18048,
        18208,
        18144,
        18304,
        17760,
        18080,
        18784,
        18368,
        17888,
        19072,
        18208,
        18240,
        17984,
        18016,
        17664,
        18400,
        18144,
        18976,
        18112,
        18016,
        17568,
        18432,
        18240,
        18272,
        17920,
        18240,
        17664,
        18400,
        17984,
        18304,
        18368,
        18656,
        17952,
        18912,
        19040,
        18496,
        18112,
        19456,
        18944,
        18304,
        18048,
        18112,
        18080,
   

In [14]:
  !curl -fsSL https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh | bash


🚀 OKC LEADERBOARD RUNNER - NVIDIA H200 SM90a

[2/4] ✓ Already signed into OKC!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 199,180 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job 2b53d9ced2b44dee
› received
› received  0:30
› queued for a GPU  0:32
› queued for a GPU  0:28
› queued for a GPU  0:58
› queued for a GPU  1:28
› queued for a GPU  1:58
› queued for a GPU  2:28
› queued for a GPU  2:58
› queued for a GPU  3:28
› queued for a GPU  3:58
› queued for a GPU  4:28
› queued for a GPU  4:58
› queued for a GPU  5:28
› queued for a GPU  5:58
› queued for a GPU  6:28
› queued for a GPU  6:58
› queued for a GPU  7:28
› queued for a GPU  7:58
› queued for a GPU  8:28
› queued for a GPU  8:58
› queued for a GPU  9:28
› running the tests, then timing against SGLang  10:29
› running the tests, then timing against SGLang  0:01
› running the tests, then timing against SGLang  0:31
› runnin

In [18]:
!curl -fsSL -H "Cache-Control: no-cache" "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh?v=$(date +%s)" | bash

✓ Already authenticated with OKC cluster!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 199,163 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job 0cac3805719e4668
› received
› received  0:30
› queued for a GPU  0:52
› queued for a GPU  0:08
› queued for a GPU  0:38
› queued for a GPU  1:08
› queued for a GPU  1:38
› queued for a GPU  2:08
› queued for a GPU  2:38
› queued for a GPU  3:08
› queued for a GPU  3:38
› queued for a GPU  4:08
› queued for a GPU  4:38
› queued for a GPU  5:08
› queued for a GPU  5:38
› queued for a GPU  6:08
› queued for a GPU  6:38
› queued for a GPU  7:08
› queued for a GPU  7:38
› queued for a GPU  8:08
› queued for a GPU  8:38
› queued for a GPU  9:08
› running the tests, then timing against SGLang  10:17
› running the tests, then timing against SGLang  0:13
› running the tests, then timing against SGLang  0:43
› running the tests, then timing against SGLa

In [19]:
!curl -fsSL -H "Cache-Control: no-cache" "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh?v=$(date +%s)" | bash

✓ Already authenticated with OKC cluster!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 199,162 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job e38a1260a3974577
› received
› received  0:30
› received  1:00
› received  1:30
› received  2:00
› received  2:30
› received  3:00
› queued for a GPU  3:19
› queued for a GPU  0:11
› queued for a GPU  0:41
› queued for a GPU  1:11
› queued for a GPU  1:41
› queued for a GPU  2:11
› queued for a GPU  2:41
› queued for a GPU  3:11
› queued for a GPU  3:41
› queued for a GPU  4:11
› queued for a GPU  4:41
› queued for a GPU  5:11
› queued for a GPU  5:41
› queued for a GPU  6:11
› running the tests, then timing against SGLang  9:38
› running the tests, then timing against SGLang  0:22
› running the tests, then timing against SGLang  0:52
› running the tests, then timing against SGLang  1:22
› running the tests, then timing against SGLang  1:52
› r

In [20]:
!curl -fsSL -H "Cache-Control: no-cache" "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh?v=$(date +%s)" | bash


✓ Already authenticated with OKC cluster!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 199,162 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job 05d19e7d7d6f4bb0
› received
› received  0:30
› received  1:00
› received  1:30
› received  2:00
› received  2:30

finished in 2:48

› muse_bf16_proj  the same code as job e38a1260a3974577  job 05d19e7d7d6f4bb0

next  okc status e38a1260a3974577
  14 of 20 runs left today, 40 of 40 checks on muse_bf16_proj

=== CLUSTER TUNE & DISABLE LOG ===
[okc] tune qkv_gate m=1: cgemv=12.19 sk2=12.96 torch=17.28 -> cgemv
[okc] tune o m=1: cgemv=8.35 sk2=8.99 torch=9.95 -> cgemv
[okc] tune pooler m=1: cgemv=16.93 sk2=17.79 torch=24.13 -> cgemv
[okc] tune unpooler m=1: cgemv=6.78 sgemv=7.1 sk2=7.52 torch=10.11 -> cgemv
[okc] tune router m=1: rgemv=6.94 sk2=8.22 torch=17.63 -> rgemv
[okc] tune qkv_gate m=16: sk2=13.02 torch=17.38 -> sk2
[okc] tune o m=16: sk2=

In [21]:
 !/root/.okc/bin/okc status e38a1260a3974577

✓ muse_bf16_proj  passed  job e38a1260a3974577

  1.476×  ━━━━━━━━━╋━━━━━─────────────  faster than SGLang  95%: 1.454–1.498×

  time        3,273.0 µs
  tests       9/9 passed
  profile     muse_bf16_proj/profiles/submission.e38a1260a3974577.txt  and .json
  output      muse_bf16_proj/profiles/submission.e38a1260a3974577.log

  CASE                    SGLANG     YOURS   SPEEDUP   
  ─────────────────────────────────────────────────────────────────
  decode 1 · qkv_gate    14.3 µs    9.1 µs    1.574×   ━━━━━━╋━━━──
  decode 1 · o            6.9 µs    5.0 µs    1.384×   ━━━━━━╋━━───
  decode 1 · pooler      21.0 µs   13.5 µs    1.554×   ━━━━━━╋━━━──
  decode 1 · unpooler     7.1 µs    3.6 µs    1.966×   ━━━━━━╋━━━━━
  decode 1 · router      10.5 µs    3.8 µs    2.796×   ━━━━━━╋━━━━━
  decode 16 · qkv_gate   13.8 µs    9.8 µs    1.415×   ━━━━━━╋━━───
  decode 16 · o           6.7 µs    5.6 µs    1.204×   ━━━━━━╋━────
  decode 16 · pooler     20.9 µs   14.3 µs    1.465×   ━━━━━━╋━━───
  d

In [22]:
 !/root/.okc/bin/okc leaderboard muse_bf16_proj

◆ muse_bf16_proj  time per decode step, lower is better

  #   NAME                                   SPEEDUP
────────────────────────────────────────────────────
  1   gogowhy                                 1.711×
  2   olety                                   1.688×
  3   dolanzhao                               1.687×
  4   reyanshsolis                            1.684×
  5   LeshengJin                              1.680×
  6   gauravjain14                       S    1.671×
  7   jt914                                   1.662×
  8   tjiang-fair                        F    1.660×
  9   sudodaksh                          S    1.653×
 10   arashattar                              1.650×
 11   Fangzhou66                              1.647×
 12   miketung                           S    1.646×
 13   mgpai22                            S    1.646×
 14   YounesElhjouji                     S    1.642×
 15   pjmullin3                          S    1.639×
 16   DTennant                            

In [23]:
!/root/.okc/bin/okc status e38a1260a3974577

✓ muse_bf16_proj  passed  job e38a1260a3974577

  1.476×  ━━━━━━━━━╋━━━━━─────────────  faster than SGLang  95%: 1.454–1.498×

  time        3,273.0 µs
  tests       9/9 passed
  profile     muse_bf16_proj/profiles/submission.e38a1260a3974577.txt  and .json
  output      muse_bf16_proj/profiles/submission.e38a1260a3974577.log

  CASE                    SGLANG     YOURS   SPEEDUP   
  ─────────────────────────────────────────────────────────────────
  decode 1 · qkv_gate    14.3 µs    9.1 µs    1.574×   ━━━━━━╋━━━──
  decode 1 · o            6.9 µs    5.0 µs    1.384×   ━━━━━━╋━━───
  decode 1 · pooler      21.0 µs   13.5 µs    1.554×   ━━━━━━╋━━━──
  decode 1 · unpooler     7.1 µs    3.6 µs    1.966×   ━━━━━━╋━━━━━
  decode 1 · router      10.5 µs    3.8 µs    2.796×   ━━━━━━╋━━━━━
  decode 16 · qkv_gate   13.8 µs    9.8 µs    1.415×   ━━━━━━╋━━───
  decode 16 · o           6.7 µs    5.6 µs    1.204×   ━━━━━━╋━────
  decode 16 · pooler     20.9 µs   14.3 µs    1.465×   ━━━━━━╋━━───
  d

In [24]:
 !curl -fsSL -H "Cache-Control: no-cache" "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh?v=$(date +%s)" | bash

✓ Already authenticated with OKC cluster!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 199,009 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job 8336556f908149e0
› received
› received  0:30
› queued for a GPU  0:32
› queued for a GPU  0:28
› queued for a GPU  0:58
› queued for a GPU  1:28
› queued for a GPU  1:58
› queued for a GPU  2:28
› queued for a GPU  2:58
› queued for a GPU  3:28
› queued for a GPU  3:58
› queued for a GPU  4:28
› queued for a GPU  4:58
› queued for a GPU  5:28
› queued for a GPU  5:58
› queued for a GPU  6:28
› queued for a GPU  6:58
› running the tests, then timing against SGLang  7:33
› running the tests, then timing against SGLang  0:27
› running the tests, then timing against SGLang  0:57
› running the tests, then timing against SGLang  1:27
› running the tests, then timing against SGLang  1:57
› running the tests, then timing against SGLang  2:27

ran for 

In [25]:
!curl -fsSL -H "Cache-Control: no-cache" "https://raw.githubusercontent.com/arunjyotisarmarsece24-jpg/neuro-spark-accel/main/run.sh?v=$(date +%s)" | bash

✓ Already authenticated with OKC cluster!

[3/4] Downloading latest Beast Kernel from GitHub...
✓ Kernel ready: 201,868 bytes

[4/4] Submitting to Meta H200 cluster & leaderboard...

✓ submitted muse_bf16_proj/submission.py to muse_bf16_proj  job 056c9688409c4484
› received
› received  0:30
› received  1:00
› received  1:30
› received  2:00
› queued for a GPU  2:26
› queued for a GPU  0:04
› queued for a GPU  0:34
› queued for a GPU  1:04
› queued for a GPU  1:34
› queued for a GPU  2:04
› queued for a GPU  2:34
› queued for a GPU  3:04
› queued for a GPU  3:34
› queued for a GPU  4:04
› queued for a GPU  4:34
› queued for a GPU  5:04
› queued for a GPU  5:34
› queued for a GPU  6:04
› queued for a GPU  6:34
› running the tests, then timing against SGLang  9:26
› running the tests, then timing against SGLang  0:04
› running the tests, then timing against SGLang  0:34
› running the tests, then timing against SGLang  1:04
› running the tests, then timing against SGLang  1:34
› running th

In [ ]:
# ==============================================================================
# 🚀 ALL-IN-ONE OKC INSTALL, AUTH, DEPLOY & SUBMIT (MUSE_BF16_PROJ)
# ==============================================================================
import os
import pty
import select
import subprocess
import time

# 1. Install okc CLI if needed and set PATH
if not os.path.exists("/root/.okc/bin/okc"):
    print("[1/5] Installing okc CLI...")
    subprocess.run("curl -fsSL https://openkernelchallenge.dev/install.sh | sh", shell=True, check=True)

os.environ["PATH"] = "/root/.okc/bin:" + os.environ["PATH"]

# 2. Check if already signed in, otherwise authenticate via Pseudo-Terminal
print("\n[2/5] Checking authentication...")
check = subprocess.run(["/root/.okc/bin/okc", "problems"], capture_output=True, text=True)

needs_login = ("not signed in" in check.stdout.lower() or "not signed in" in check.stderr.lower() or check.returncode != 0)

if needs_login:
    print("[*] Starting GitHub Device Flow Login...")
    master, slave = pty.openpty()
    proc = subprocess.Popen(
        ["/root/.okc/bin/okc", "login"],
        stdin=slave, stdout=slave, stderr=slave, close_fds=True
    )
    os.close(slave)

    # Wait 1.0 second and send 'y\n' through the real PTY
    time.sleep(1.0)
    os.write(master, b"y\n")

    # Read output and display device code
    while True:
        r, _, _ = select.select([master], [], [], 2.0)
        if r:
            try:
                data = os.read(master, 1024)
                if not data:
                    break
                print(data.decode("utf-8", errors="ignore"), end="", flush=True)
            except OSError:
                break
        elif proc.poll() is not None:
            break

    proc.wait()
    print("\n✓ Authentication completed!")
else:
    print("✓ Already authenticated!")

# 3. Initialize muse_bf16_proj workspace
print("\n[3/5] Initializing muse_bf16_proj workspace...")
subprocess.run(["/root/.okc/bin/okc", "init", "muse_bf16_proj"])

# 4. Deploy our optimized Hopper H200 kernel into muse_bf16_proj/submission.py
print("\n[4/5] Deploying optimized Hopper H200 kernel to muse_bf16_proj/submission.py...")
optimized_code = """# okc: problem=muse_bf16_proj
\"\"\"Optimized Dense Projections for Muse Spark Open on Hopper H200.
Author: Arunjyoti Sarma (IIT BHU Research Lab)

Key Techniques:
1. Zero-Allocation Hot Path: Pre-slice and pre-contiguous all weight buffers during untimed prepare().
2. Pre-Transposed Layouts: Eliminate runtime matrix transpositions.
3. Dual-Path Router: Pre-transposed FP32 buffer for tokens <= 8, split BF16 GEMM for tokens > 8.
4. Native cuBLASLt Dispatch with Zero Slicing Overhead.
\"\"\"

from typing import TypedDict
import torch
import torch.nn.functional as F

try:
    from sglang.srt.layers.quantization.unquant import _bf16_gemm_dispatch_impl
    HAS_SGLANG = True
except Exception:
    HAS_SGLANG = False
    _bf16_gemm_dispatch_impl = None


class Weights(TypedDict):
    qkv_gate: torch.Tensor  # bfloat16 [2304, 8192]
    o: torch.Tensor         # bfloat16 [8192, 1024]
    pooler: torch.Tensor    # bfloat16 [4096, 8192]
    unpooler: torch.Tensor  # bfloat16 [8192, 4096]
    router: torch.Tensor    # float32 [256, 8192]


class Config(TypedDict):
    m: int
    n: int
    k: int
    layer: str


class Inputs(TypedDict):
    x: torch.Tensor
    config: Config


ROUTER_FP32_MAX_TOKENS = 8


def prepare(weights: Weights):
    state = {}
    for name, w in weights.items():
        w_contig = w.contiguous()
        state[name] = w_contig
        state[f"{name}_t"] = w_contig.t().contiguous()

    unpooler = weights["unpooler"]
    state["unpooler_1024"] = unpooler[:1024].contiguous()
    state["unpooler_1024_t"] = unpooler[:1024].t().contiguous()
    state["unpooler_8192"] = unpooler.contiguous()
    state["unpooler_8192_t"] = unpooler.t().contiguous()

    router = weights["router"]
    state["router_t"] = router.t().contiguous()
    hi = router.to(torch.bfloat16)
    lo = (router - hi.float()).to(torch.bfloat16)
    state["router_hi_t"] = hi.t().contiguous()
    state["router_lo_t"] = lo.t().contiguous()

    return state


def kernel_fn(data: Inputs, state: dict) -> torch.Tensor:
    x = data["x"]
    cfg = data["config"]
    layer = cfg["layer"]

    if layer == "router":
        if x.shape[0] <= ROUTER_FP32_MAX_TOKENS:
            return torch.mm(x.float(), state["router_t"])
        acc = torch.mm(x, state["router_hi_t"], out_dtype=torch.float32)
        acc += torch.mm(x, state["router_lo_t"], out_dtype=torch.float32)
        return acc

    if layer == "unpooler":
        n = cfg["n"]
        if n == 1024:
            w = state["unpooler_1024"]
            w_t = state["unpooler_1024_t"]
        elif n == 8192:
            w = state["unpooler_8192"]
            w_t = state["unpooler_8192_t"]
        else:
            w = state["unpooler"][:n]
            w_t = w.t().contiguous()

        if HAS_SGLANG and _bf16_gemm_dispatch_impl is not None:
            return _bf16_gemm_dispatch_impl(x, w, None)
        return torch.mm(x, w_t)

    w = state[layer]
    if HAS_SGLANG and _bf16_gemm_dispatch_impl is not None:
        return _bf16_gemm_dispatch_impl(x, w, None)
    return torch.mm(x, state[f"{layer}_t"])
"""

os.makedirs("muse_bf16_proj", exist_ok=True)
with open("muse_bf16_proj/submission.py", "w") as f:
    f.write(optimized_code)
print("✓ muse_bf16_proj/submission.py ready!")

# 5. Submit to Meta's H200 cluster
print("\n[5/5] Submitting to Meta's H200 cluster & leaderboard...\n")
subprocess.run(["/root/.okc/bin/okc", "submit", "muse_bf16_proj/submission.py"])
